# B3b Defense – 04 XGBoost Training

**Objective:** Train XGBoost on `defense_feature_matrix.csv` using `TimeSeriesSplit(n_splits=5)`.
Report MAE, RMSE, sMAPE, and WMAPE per fold. Save the best model to disk.

In [1]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

In [2]:
DATA_PROCESSED = '../data/processed/'
MODELS_DIR     = '../models/'

os.makedirs(MODELS_DIR, exist_ok=True)

In [3]:
# Load feature matrix
df = pd.read_csv(DATA_PROCESSED + 'defense_feature_matrix.csv', parse_dates=['date'], index_col='date')

# Define feature columns and target
feature_cols = [col for col in df.columns if col != 'ADEFNO']
target_col   = 'ADEFNO'

X = df[feature_cols]
y = df[target_col]

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape:         {y.shape}')
print(f'Features: {feature_cols}')

Feature matrix shape: (305, 31)
Target shape:         (305,)
Features: ['ADEFNO_diff', 'IPB52300S', 'IPB52300S_diff', 'FDEFX', 'FDEFX_diff', 'ADEFNO_diff_lag_1', 'ADEFNO_diff_lag_2', 'ADEFNO_diff_lag_3', 'ADEFNO_diff_lag_4', 'ADEFNO_diff_lag_5', 'ADEFNO_diff_lag_6', 'IPB52300S_diff_lag_1', 'IPB52300S_diff_lag_2', 'IPB52300S_diff_lag_3', 'IPB52300S_diff_lag_4', 'IPB52300S_diff_lag_5', 'IPB52300S_diff_lag_6', 'FDEFX_diff_lag_1', 'FDEFX_diff_lag_2', 'FDEFX_diff_lag_3', 'adefno_lag_1', 'adefno_lag_2', 'adefno_lag_3', 'adefno_rolling_3m_mean', 'adefno_rolling_3m_std', 'adefno_rolling_6m_mean', 'adefno_rolling_6m_std', 'month', 'quarter', 'year', 'is_q4']


In [4]:
def smape(actual, pred):
    """Symmetric Mean Absolute Percentage Error (in %)"""
    numerator   = 2 * np.abs(actual - pred)
    denominator = np.abs(actual) + np.abs(pred)
    # Avoid division by zero
    mask = denominator != 0
    return np.mean(numerator[mask] / denominator[mask]) * 100


def wmape(actual, pred):
    """Weighted Mean Absolute Percentage Error (in %)"""
    return np.sum(np.abs(actual - pred)) / np.sum(np.abs(actual)) * 100

In [5]:
tscv = TimeSeriesSplit(n_splits=5)

fold_results = []
best_model   = None

for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = xgb.XGBRegressor(
        n_estimators  = 200,
        max_depth      = 4,
        learning_rate  = 0.05,
        subsample      = 0.8,
        random_state   = 42,
        verbosity      = 0
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae   = mean_absolute_error(y_test, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
    smape_val = smape(y_test.values, y_pred)
    wmape_val = wmape(y_test.values, y_pred)

    fold_results.append({
        'Fold':  fold_idx,
        'MAE':   round(mae, 4),
        'RMSE':  round(rmse, 4),
        'sMAPE': round(smape_val, 2),
        'WMAPE': round(wmape_val, 2)
    })

    print(f'Fold {fold_idx}: MAE={mae:.4f}  RMSE={rmse:.4f}  sMAPE={smape_val:.2f}%  WMAPE={wmape_val:.2f}%')

    # Keep the last fold's model as the final model
    best_model = model

print('\nCross-validation complete.')

Fold 1: MAE=1258.4795  RMSE=1709.6061  sMAPE=14.31%  WMAPE=14.35%
Fold 2: MAE=1265.5778  RMSE=1562.1766  sMAPE=12.58%  WMAPE=12.49%
Fold 3: MAE=870.9164  RMSE=1247.4234  sMAPE=8.97%  WMAPE=9.12%
Fold 4: MAE=1068.9883  RMSE=1592.6371  sMAPE=8.65%  WMAPE=8.64%
Fold 5: MAE=1149.6987  RMSE=1750.1409  sMAPE=8.09%  WMAPE=8.63%

Cross-validation complete.


In [6]:
results_df = pd.DataFrame(fold_results).set_index('Fold')
print(results_df)
print('\n--- Mean across folds ---')
print(results_df.mean().round(4))

            MAE       RMSE  sMAPE  WMAPE
Fold                                    
1     1258.4795  1709.6061  14.31  14.35
2     1265.5778  1562.1766  12.58  12.49
3      870.9164  1247.4234   8.97   9.12
4     1068.9883  1592.6371   8.65   8.64
5     1149.6987  1750.1409   8.09   8.63

--- Mean across folds ---
MAE      1122.7321
RMSE     1572.3968
sMAPE      10.5200
WMAPE      10.6460
dtype: float64


In [7]:
model_path = MODELS_DIR + 'xgboost_defense_market.pkl'
joblib.dump(best_model, model_path)
print(f'Model saved to: {model_path}')

Model saved to: ../models/xgboost_defense_market.pkl


## Performance Assessment

**Data basis:** 305 observations, 31 features, target ADEFNO in USD millions.

| Fold | MAE (USD mn) | RMSE (USD mn) | sMAPE | WMAPE |
|------|-------------|--------------|-------|-------|
| 1 | 1,258 | 1,710 | 14.31 % | 14.35 % |
| 2 | 1,266 | 1,562 | 12.58 % | 12.49 % |
| 3 | 871 | 1,247 | 8.97 % | 9.12 % |
| 4 | 1,069 | 1,593 | 8.65 % | 8.64 % |
| 5 | 1,150 | 1,750 | 8.09 % | 8.63 % |
| **Mean** | **1,123** | **1,572** | **10.52 %** | **10.65 %** |

### Positive signals

- **10.5 % sMAPE is solid for defense market forecasting.** Defense orders are structurally
  volatile — large contracts cause spikes that no model can anticipate exactly. For reference,
  the B3-Reference model (b3a_compressors) achieved ~17 % sMAPE on only 33 training months.
  With 305 observations available here, the results are considerably more stable.

- **sMAPE ≈ WMAPE across all folds.** The close agreement between both metrics indicates that
  no single extreme outlier is distorting the error figures — the error distribution is consistent.

- **Clear learning trend across folds:** sMAPE drops from 14.3 % (Fold 1) to 8.1 % (Fold 5).
  This is the expected behavior with TimeSeriesSplit — the more training history available,
  the better the model performs. The model benefits noticeably from additional data.

### Limitations

- **RMSE/MAE ratio ≈ 1.4.** A value clearly above 1.0 points to occasional larger individual
  errors — typical for lumpy large contracts in the defense sector that the model cannot foresee.

- **Fold 5 has the best sMAPE but the second-highest RMSE.** The model stays close to the target
  in percentage terms but misses individual months with high absolute amounts more strongly.
  For a market volume forecast this is acceptable — the trend is captured, individual peaks are not.

- **No hyperparameter tuning applied.** A grid or random search over `max_depth`, `n_estimators`,
  and `learning_rate` could yield further improvement, but is not warranted for a prototype
  operating on macro-level data.

### Conclusion

The model is well-suited for its intended purpose: estimating the addressable market volume in
a new segment as a data-backed benchmark for management input. It is not optimized for
tick-level precision but provides an order-of-magnitude forecast that is further processed
with market share scenarios in SAC (`Segment_Forecast = ADEFNO_forecast × market_share`).